In [ ]:
from matplotlib import pyplot as plt
from sklearn.discriminant_analysis import StandardScaler
from sklearn.metrics import silhouette_score
import streamlit as st
import pandas as pd 
import numpy as np 
from nba_api.stats.endpoints import LeagueGameLog
from nba_api.stats.endpoints import teamgamelogs

from sklearn.cluster import KMeans
import datetime
import os

In [ ]:
from nba_api.stats.endpoints import LeagueDashTeamPtShot
from nba_api.stats.endpoints import LeagueHustleStatsTeam
from nba_api.stats.endpoints import LeagueDashOppPtShot

In [ ]:
gen_opp_def = leaguedashteamstats.LeagueDashTeamStats(per_mode_detailed="Per100Possessions", measure_type_detailed_defense="Opponent", league_id_nullable="00")
gen_opp_def_stats = gen_opp_def.get_data_frames()[0]
gen_opp_def_stats.columns
print(gen_opp_def_stats.columns
, 'gen opp defense , per 100')
gen_opp_def_stats = gen_opp_def_stats[["TEAM_ID",
"TEAM_NAME",
"OPP_TOV"
]]


gen_opp_def_extra = leaguedashteamstats.LeagueDashTeamStats(per_mode_detailed="Per100Possessions", measure_type_detailed_defense="Defense", league_id_nullable="00")
gen_opp_def_extra_stats = gen_opp_def_extra.get_data_frames()[0]
gen_opp_def_extra_stats.columns
print(gen_opp_def_extra_stats.columns
, 'gen opp defense extra , per 100')
gen_opp_def_extra_stats = gen_opp_def_extra_stats[["TEAM_ID",
"TEAM_NAME",
"DEF_RATING",
"OPP_PTS_OFF_TOV",
"OPP_PTS_2ND_CHANCE",
"OPP_PTS_FB",
"OPP_PTS_PAINT"
]]
gen_opp_def_four_factors = leaguedashteamstats.LeagueDashTeamStats(per_mode_detailed="Per100Possessions", measure_type_detailed_defense="Four Factors", league_id_nullable="00")
gen_opp_def_four_factors_stats = gen_opp_def_four_factors.get_data_frames()[0]
gen_opp_def_four_factors_stats.columns


In [ ]:
gen_opp_def_four_factors_stats = gen_opp_def_four_factors_stats[["TEAM_ID",
"TEAM_NAME",
"EFG_PCT",
"FTA_RATE",
"TM_TOV_PCT",
"OREB_PCT",
"OPP_EFG_PCT",
"OPP_FTA_RATE",
"OPP_TOV_PCT",
"OPP_OREB_PCT"
]]

gen_opp_def_four_factors_stats

hustle = LeagueHustleStatsTeam(per_mode_time="PerGame", season="2025-26")

hustle_stats = hustle.get_data_frames()[0]
hustle_stats.columns
print(hustle_stats.columns
, 'hustle stats , per game')


hustle_stats = hustle_stats[["TEAM_ID",
"TEAM_NAME",
"CONTESTED_SHOTS",
"CONTESTED_SHOTS_3PT",
"DEFLECTIONS",
"CHARGES_DRAWN",
"DEF_BOXOUTS",
"DEF_LOOSE_BALLS_RECOVERED"
]]





In [ ]:
advanced_nba = leaguedashteamstats.LeagueDashTeamStats(per_mode_detailed="Per100Possessions", measure_type_detailed_defense="Advanced", league_id_nullable="00")
advanced_nba_stats = advanced_nba.get_data_frames()[0]
advanced_nba_stats.columns
print(advanced_nba_stats.columns, 'adv nba , per 100')
advanced_nba_stats = advanced_nba_stats[["TEAM_ID",
"TEAM_NAME",
"OFF_RATING",
"DEF_RATING",
"NET_RATING",
"AST_PCT",
"TM_TOV_PCT",
"EFG_PCT",
"TS_PCT",
"PACE",
"PIE"
]]
misc_nba = leaguedashteamstats.LeagueDashTeamStats(per_mode_detailed="Per100Possessions", measure_type_detailed_defense="Misc", league_id_nullable="00")
misc_nba_stats = misc_nba.get_data_frames()[0]
misc_nba_stats.columns
print(misc_nba_stats.columns , 'misc , per 100')
misc_nba_stats = misc_nba_stats[["TEAM_ID",
"TEAM_NAME",
"PTS_OFF_TOV",
"PTS_2ND_CHANCE",
"PTS_FB",
"PTS_PAINT",
"OPP_PTS_OFF_TOV",
"OPP_PTS_2ND_CHANCE",
"OPP_PTS_FB",
"OPP_PTS_PAINT"
]]
shot_def = (LeagueDashOppPtShot(season_type_all_star="Regular Season", per_mode_simple="PerGame" ))
opp_shot_splits = shot_def.get_data_frames()[0]
print(opp_shot_splits.columns)
print('opp shots , per game')
opp_shot_splits = opp_shot_splits[["TEAM_ID",
"TEAM_NAME",
"TEAM_ABBREVIATION",
"FGA_FREQUENCY",
"FG2A_FREQUENCY",
"FG3A_FREQUENCY",
"EFG_PCT"
]]
pergame_full_table = opp_shot_splits.merge(hustle_stats, 
                                    on="TEAM_ID",
                                    how="inner")


In [ ]:
normal_pergame_table_2 = pergame_full_table.merge(
    advanced_nba_stats,
    on="TEAM_ID",
    how='inner'
)

normal_pergame_table_2.columns

cols_to_drop = ["TEAM_NAME_x",
"TEAM_NAME_y",
"EFG_PCT_y"
]

normal_pergame_table = normal_pergame_table_2.drop(columns=cols_to_drop)


normal_pergame_table


In [ ]:
hustle_cols = [
    "CONTESTED_SHOTS",
    "CONTESTED_SHOTS_3PT",
    "DEFLECTIONS",
    "CHARGES_DRAWN",
    "DEF_BOXOUTS",
    "DEF_LOOSE_BALLS_RECOVERED"
]

for col in hustle_cols:
    normal_pergame_table[f"{col}_PER100"] = normal_pergame_table[col] / normal_pergame_table["PACE"] * 100



cols_to_drop = ["CONTESTED_SHOTS",
"CONTESTED_SHOTS_3PT",
"DEFLECTIONS",
"CHARGES_DRAWN",
"DEF_BOXOUTS",
"DEF_LOOSE_BALLS_RECOVERED"
]


normal_pergame_table = normal_pergame_table.drop(columns=cols_to_drop)

normal_pergame_table

hustle_adv_per100 = normal_pergame_table.copy(deep=True)

general_stats = gen_opp_def_extra_stats.merge(
    gen_opp_def_four_factors_stats.drop(columns=["TEAM_NAME"]),
    on="TEAM_ID",
    how="inner"
)

general_stats

adv_and_gen_stats = general_stats.merge(
    hustle_adv_per100.drop(columns=["TEAM_NAME"]),
    on="TEAM_ID",
    how="inner"
)

adv_gen_misc_stats = adv_and_gen_stats.merge(misc_nba_stats.drop(columns=["TEAM_NAME"]),
    on="TEAM_ID",
    how="inner")

    cols_to_drop = [
    # --- merge artifacts / duplicates ---
    "DEF_RATING_x",
    "DEF_RATING_y",
    "TM_TOV_PCT_x",
    "TM_TOV_PCT_y",
    "EFG_PCT_x",

    # --- duplicate opponent scoring breakdown ---
    "OPP_PTS_OFF_TOV_x",
    "OPP_PTS_2ND_CHANCE_x",
    "OPP_PTS_FB_x",
    "OPP_PTS_PAINT_x",

    # --- non-feature identifiers (add back later if needed) ---
    "TEAM_NAME",
    "TEAM_ABBREVIATION"
]

merged_df = adv_gen_misc_stats.drop(columns=cols_to_drop, errors="ignore")

merged_df.drop(columns=['FGA_FREQUENCY'])

merged_df.to_parquet('newdefensedata.parquet')
